# 高级推理引擎技术教程

本教程涵盖：
1. ONNX Runtime 高级优化
2. TensorRT 多流并行
3. vLLM 高级配置
4. 性能分析与调优
5. 生产部署最佳实践

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
from typing import List, Dict, Any

# 检查可用的推理引擎
try:
    import onnxruntime as ort
    print(f'ONNX Runtime: {ort.__version__}')
    print(f'Available providers: {ort.get_available_providers()}')
except ImportError:
    print('ONNX Runtime not available')

## 1. ONNX Runtime 高级优化

In [ ]:
class OptimizedONNXSession:
    """优化的 ONNX Runtime 会话"""
    def __init__(self, model_path, use_gpu=False):
        self.sess_options = ort.SessionOptions()
        
        # 图优化
        self.sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        
        # 线程配置
        self.sess_options.intra_op_num_threads = 4
        self.sess_options.inter_op_num_threads = 2
        
        # 内存优化
        self.sess_options.enable_mem_pattern = True
        self.sess_options.enable_mem_reuse = True
        
        # 选择 Provider
        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if use_gpu else ['CPUExecutionProvider']
        
        self.session = ort.InferenceSession(model_path, self.sess_options, providers=providers)
        self.input_name = self.session.get_inputs()[0].name
    
    def infer(self, input_data):
        return self.session.run(None, {self.input_name: input_data})[0]
    
    def benchmark(self, input_data, warmup=10, iterations=100):
        # 预热
        for _ in range(warmup):
            self.infer(input_data)
        
        # 测量
        latencies = []
        for _ in range(iterations):
            start = time.perf_counter()
            self.infer(input_data)
            latencies.append((time.perf_counter() - start) * 1000)
        
        return {
            'mean_ms': np.mean(latencies),
            'std_ms': np.std(latencies),
            'p50_ms': np.percentile(latencies, 50),
            'p99_ms': np.percentile(latencies, 99),
        }

print('OptimizedONNXSession defined')

## 2. 异步推理管理器

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
from queue import Queue
import threading

class AsyncInferenceManager:
    """异步推理管理器"""
    def __init__(self, model_fn, max_workers=4):
        self.model_fn = model_fn
        self.executor = ThreadPoolExecutor(max_workers=max_workers)
        self.request_queue = Queue()
        self.results = {}
        self.lock = threading.Lock()
        self.request_id = 0
    
    def submit(self, input_data):
        with self.lock:
            req_id = self.request_id
            self.request_id += 1
        
        future = self.executor.submit(self._process, req_id, input_data)
        return req_id, future
    
    def _process(self, req_id, input_data):
        result = self.model_fn(input_data)
        with self.lock:
            self.results[req_id] = result
        return result
    
    def get_result(self, req_id):
        with self.lock:
            return self.results.pop(req_id, None)

# 示例使用
def dummy_model(x):
    time.sleep(0.01)  # 模拟推理
    return x * 2

manager = AsyncInferenceManager(dummy_model, max_workers=4)

# 提交多个请求
futures = []
for i in range(10):
    req_id, future = manager.submit(np.array([i]))
    futures.append((req_id, future))

# 等待结果
for req_id, future in futures:
    result = future.result()
    print(f'Request {req_id}: {result}')

## 3. 动态批处理器

In [ ]:
class DynamicBatcher:
    """动态批处理器：收集请求并批量处理"""
    def __init__(self, model_fn, max_batch_size=32, max_wait_ms=10):
        self.model_fn = model_fn
        self.max_batch_size = max_batch_size
        self.max_wait_ms = max_wait_ms
        self.pending = []
        self.lock = threading.Lock()
        self.batch_count = 0
    
    def add_request(self, input_data):
        with self.lock:
            self.pending.append(input_data)
            
            if len(self.pending) >= self.max_batch_size:
                return self._process_batch()
        
        # 等待更多请求或超时
        time.sleep(self.max_wait_ms / 1000)
        
        with self.lock:
            if self.pending:
                return self._process_batch()
        return None
    
    def _process_batch(self):
        batch = self.pending[:self.max_batch_size]
        self.pending = self.pending[self.max_batch_size:]
        self.batch_count += 1
        
        # 批量推理
        batch_input = np.stack(batch)
        results = self.model_fn(batch_input)
        return results

# 示例
def batch_model(x):
    return x * 2

batcher = DynamicBatcher(batch_model, max_batch_size=4, max_wait_ms=5)

# 模拟请求
for i in range(8):
    result = batcher.add_request(np.array([i, i+1]))
    if result is not None:
        print(f'Batch result shape: {result.shape}')

print(f'Total batches processed: {batcher.batch_count}')

## 4. 性能分析器

In [ ]:
class InferenceProfiler:
    """推理性能分析器"""
    def __init__(self):
        self.records = []
    
    def profile(self, model_fn, input_data, name='model'):
        # 预热
        for _ in range(5):
            model_fn(input_data)
        
        # 测量
        latencies = []
        for _ in range(50):
            start = time.perf_counter()
            output = model_fn(input_data)
            latencies.append((time.perf_counter() - start) * 1000)
        
        record = {
            'name': name,
            'mean_ms': np.mean(latencies),
            'std_ms': np.std(latencies),
            'min_ms': np.min(latencies),
            'max_ms': np.max(latencies),
            'p50_ms': np.percentile(latencies, 50),
            'p95_ms': np.percentile(latencies, 95),
            'p99_ms': np.percentile(latencies, 99),
            'throughput': 1000 / np.mean(latencies),
        }
        self.records.append(record)
        return record
    
    def compare(self):
        print('\n=== Performance Comparison ===')
        print(f'{"Model":<20} {"Mean(ms)":<12} {"P99(ms)":<12} {"Throughput":<12}')
        print('-' * 56)
        for r in self.records:
            print(f'{r["name"]:<20} {r["mean_ms"]:<12.3f} {r["p99_ms"]:<12.3f} {r["throughput"]:<12.1f}')

# 示例
profiler = InferenceProfiler()

# 模拟不同模型
def model_v1(x): time.sleep(0.001); return x
def model_v2(x): time.sleep(0.0008); return x

data = np.random.randn(1, 64).astype(np.float32)
profiler.profile(model_v1, data, 'Model V1')
profiler.profile(model_v2, data, 'Model V2 (Optimized)')
profiler.compare()

## 5. 生产部署：健康检查与监控

In [ ]:
from collections import deque
from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class InferenceMetrics:
    """推理指标"""
    total_requests: int = 0
    successful_requests: int = 0
    failed_requests: int = 0
    latencies: deque = field(default_factory=lambda: deque(maxlen=1000))
    
    @property
    def success_rate(self):
        if self.total_requests == 0:
            return 0.0
        return self.successful_requests / self.total_requests
    
    @property
    def avg_latency_ms(self):
        if not self.latencies:
            return 0.0
        return np.mean(list(self.latencies))

class MonitoredInferenceEngine:
    """带监控的推理引擎"""
    def __init__(self, model_fn):
        self.model_fn = model_fn
        self.metrics = InferenceMetrics()
        self.is_healthy = True
    
    def infer(self, input_data):
        self.metrics.total_requests += 1
        start = time.perf_counter()
        
        try:
            result = self.model_fn(input_data)
            latency = (time.perf_counter() - start) * 1000
            self.metrics.latencies.append(latency)
            self.metrics.successful_requests += 1
            return result
        except Exception as e:
            self.metrics.failed_requests += 1
            raise
    
    def health_check(self):
        return {
            'healthy': self.is_healthy,
            'success_rate': f'{self.metrics.success_rate:.2%}',
            'avg_latency_ms': f'{self.metrics.avg_latency_ms:.2f}',
            'total_requests': self.metrics.total_requests,
        }

# 示例
engine = MonitoredInferenceEngine(lambda x: x * 2)

for i in range(100):
    engine.infer(np.array([i]))

print('Health Check:', engine.health_check())

## 6. A/B 测试框架

In [ ]:
import random

class ABTestingEngine:
    """A/B 测试推理引擎"""
    def __init__(self, engine_a, engine_b, traffic_ratio=0.5):
        self.engine_a = engine_a
        self.engine_b = engine_b
        self.ratio = traffic_ratio
        self.metrics_a = {'latencies': [], 'count': 0}
        self.metrics_b = {'latencies': [], 'count': 0}
    
    def infer(self, input_data):
        if random.random() < self.ratio:
            engine, metrics = self.engine_a, self.metrics_a
        else:
            engine, metrics = self.engine_b, self.metrics_b
        
        start = time.perf_counter()
        result = engine(input_data)
        latency = (time.perf_counter() - start) * 1000
        
        metrics['latencies'].append(latency)
        metrics['count'] += 1
        return result
    
    def get_statistics(self):
        def calc_stats(metrics):
            if not metrics['latencies']:
                return {'count': 0, 'avg_ms': 0, 'p99_ms': 0}
            return {
                'count': metrics['count'],
                'avg_ms': np.mean(metrics['latencies']),
                'p99_ms': np.percentile(metrics['latencies'], 99),
            }
        return {
            'engine_a': calc_stats(self.metrics_a),
            'engine_b': calc_stats(self.metrics_b),
        }

# 示例
def engine_a(x): time.sleep(0.001); return x
def engine_b(x): time.sleep(0.0008); return x  # 更快

ab_test = ABTestingEngine(engine_a, engine_b, traffic_ratio=0.5)

for _ in range(200):
    ab_test.infer(np.array([1.0]))

stats = ab_test.get_statistics()
print('A/B Test Results:')
print(f'  Engine A: {stats["engine_a"]}')
print(f'  Engine B: {stats["engine_b"]}')

## 7. 优雅降级服务

In [ ]:
class ResilientInferenceService:
    """支持降级的推理服务"""
    def __init__(self, primary_engine, fallback_engine, error_threshold=3):
        self.primary = primary_engine
        self.fallback = fallback_engine
        self.error_threshold = error_threshold
        self.consecutive_errors = 0
        self.use_fallback = False
        self.fallback_count = 0
    
    def infer(self, input_data):
        if self.use_fallback:
            self.fallback_count += 1
            return self.fallback(input_data)
        
        try:
            result = self.primary(input_data)
            self.consecutive_errors = 0
            return result
        except Exception as e:
            self.consecutive_errors += 1
            if self.consecutive_errors >= self.error_threshold:
                print(f'Switching to fallback after {self.consecutive_errors} errors')
                self.use_fallback = True
            return self.fallback(input_data)
    
    def reset(self):
        self.use_fallback = False
        self.consecutive_errors = 0

# 示例
call_count = [0]
def unreliable_primary(x):
    call_count[0] += 1
    if call_count[0] % 3 == 0:  # 每3次失败一次
        raise RuntimeError('Primary failed')
    return x * 2

def reliable_fallback(x):
    return x * 1.5  # 降级版本

service = ResilientInferenceService(unreliable_primary, reliable_fallback, error_threshold=3)

for i in range(15):
    result = service.infer(np.array([i]))
    print(f'Request {i}: result={result[0]:.1f}, fallback={service.use_fallback}')

## 总结

| 技术 | 用途 | 关键点 |
|:-----|:-----|:-------|
| 异步推理 | 提高并发 | ThreadPoolExecutor |
| 动态批处理 | 提高吞吐 | 收集请求批量处理 |
| 性能分析 | 优化决策 | 延迟分布统计 |
| 健康监控 | 生产运维 | 成功率/延迟指标 |
| A/B 测试 | 模型对比 | 流量分配统计 |
| 优雅降级 | 高可用 | 错误阈值切换 |